# Notebook principal — INF6243

Ce notebook appelle les scripts du dossier `Code/` et affiche les résultats.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from IPython.display import display, Markdown, Image

from main import run_pipeline

In [ ]:
# Configuration d'exécution (modifiable facilement)
RUN_CONFIG = {
    "max_samples": 12000,      # mettre None pour utiliser 100% des données
    "distilbert_epochs": 1,    # augmenter (ex: 2-3) si vous voulez entraîner DistilBERT plus longtemps
}

# Exécute toutes les étapes : EDA, entraînement, évaluation, sauvegardes
artifacts = run_pipeline(
    max_samples=RUN_CONFIG["max_samples"],
    distilbert_epochs=RUN_CONFIG["distilbert_epochs"],
)
artifacts

In [ ]:
# Lecture du rapport JSON produit par le pipeline
report_path = Path(artifacts["report_path"])
with report_path.open("r", encoding="utf-8") as file:
    report = json.load(file)

best_name = report.get("best_model", "N/A")
best_test = report.get("best_model_test_metrics", {})
selection_method = report.get("model_selection_method", {})
ranking = report.get("model_selection_ranking", [])
run_config = report.get("run_config", {})

print("Meilleur modèle:", best_name)
print("Métriques test:", best_test)
print("Classement global:", ranking)
print("Formule de sélection:", selection_method.get("formula", "non disponible"))
print("DistilBERT:", report.get("distilbert_note", "information non disponible"))
print("Configuration d'exécution:", run_config)

if report.get("best_model_selection_explanation"):
    display(Markdown("**Justification:** " + report["best_model_selection_explanation"]))

In [ ]:
# Tableau comparatif complet (robuste même si certaines clés sont absentes)
rows = []
all_models = report.get("all_models", {})
selection_scores = report.get("model_selection_scores", {})

for model_name, payload in all_models.items():
    row = {"model": model_name}
    row.update(payload.get("test_metrics", {}))
    row["selection_score"] = selection_scores.get(model_name)
    row["cv_f1_macro_mean"] = payload.get("cv_f1_macro_mean")
    row["why_model"] = payload.get("model_why", "")
    rows.append(row)

comparison_df = pd.DataFrame(rows).sort_values("selection_score", ascending=False)
comparison_df

## Pourquoi ce meilleur modèle ?

La sélection est faite par un score global pondéré:

`selection_score = 0.35 * val_f1_macro + 0.40 * test_f1_macro + 0.25 * cv_f1_macro_mean`

- Le **test F1** a le poids principal (généralisation réelle).
- Le **validation F1** confirme la qualité pendant le tuning.
- La **moyenne CV** mesure la stabilité.
- Pour un modèle deep learning coûteux (ex: DistilBERT), la composante CV peut être approchée par le score validation (cf. `cv_fallback_for_models`).

### Pourquoi `best_cv_score` est `NaN` pour DistilBERT ?

- Les modèles classiques utilisent `GridSearchCV`, donc `best_cv_score` existe.
- DistilBERT est entraîné en **fine-tuning direct** (pas de `GridSearchCV` complet, trop coûteux en temps/ressources).
- Donc `best_cv_score` reste vide (`NaN`) par design, et la stabilité est reflétée dans `cv_fallback_for_models` + score de validation.

Le détail est dans `Outputs/reports/metrics_report.json` (`model_selection_method`, `model_selection_scores`, `model_selection_ranking`).

## Où sont les résultats ?

- Figures: `Outputs/figures/`
  - `models_compilation_overview.png` (figure synthèse comparative)
  - `confusion_matrices_all_models.png` (toutes les matrices de confusion)
- Rapport JSON: `Outputs/reports/metrics_report.json`
- Modèle final:
  - `Outputs/models/best_model.joblib` (modèles sklearn)
  - ou note JSON si le meilleur est deep learning

In [ ]:
# Détails des hyperparamètres retenus par modèle
# Note: DistilBERT n'utilise pas GridSearchCV complet, donc best_cv_score peut rester vide (NaN).
tuning_rows = []
for model_name, payload in report.get("all_models", {}).items():
    tuning = payload.get("tuning", {})
    tuning_rows.append(
        {
            "model": model_name,
            "status": payload.get("status"),
            "best_cv_score": tuning.get("best_cv_score"),
            "cv_note": tuning.get("note"),
            "best_params": str(tuning.get("best_params", {})),
        }
    )

pd.DataFrame(tuning_rows).sort_values("model")

In [ ]:
# Affichage des figures de synthèse comparative
fig_dir = Path(artifacts["figures_dir"])
for fig_name in ["models_compilation_overview.png", "confusion_matrices_all_models.png"]:
    fig_path = fig_dir / fig_name
    if fig_path.exists():
        display(Markdown(f"### {fig_name}"))
        display(Image(filename=str(fig_path)))
    else:
        print(f"Figure absente: {fig_path}")

## Couverture complete des modeles

Le tableau ci-dessous affiche **tous les modeles attendus**, y compris ceux non executes, avec leur statut et la raison (si ignore/erreur).

In [ ]:
# Tableau global: statut d'execution + metriques + hyperparametres + features
rows = []
for model_name in report.get("expected_models", []):
    payload = report.get("all_models", {}).get(model_name, {})
    rows.append(
        {
            "model": model_name,
            "status": payload.get("status", "unknown"),
            "error_or_reason": payload.get("error"),
            "selection_score": payload.get("selection_score"),
            "val_f1_macro": payload.get("validation_metrics", {}).get("f1_macro"),
            "test_f1_macro": payload.get("test_metrics", {}).get("f1_macro"),
            "cv_f1_macro_mean": payload.get("cv_f1_macro_mean"),
            "representation": payload.get("feature_config", {}).get("representation"),
            "tfidf_ngram_range": payload.get("feature_config", {}).get("tfidf_ngram_range"),
            "tfidf_min_df": payload.get("feature_config", {}).get("tfidf_min_df"),
            "tfidf_max_features": payload.get("feature_config", {}).get("tfidf_max_features"),
            "deep_model_name": payload.get("feature_config", {}).get("deep_model_name"),
            "best_params": str(payload.get("tuning", {}).get("best_params", {})),
        }
    )

full_models_df = pd.DataFrame(rows)
full_models_df

In [ ]:
# Resume lisible des modeles non actifs
inactive = full_models_df[full_models_df["status"] != "trained"]
if inactive.empty:
    print("Tous les modeles attendus ont ete entraines.")
else:
    print("Modeles non actifs (skipped/failed):")
    display(inactive[["model", "status", "error_or_reason"]])

print("\nArtefacts principaux generes:")
print(report.get("artifacts_generated", {}))